# Live demo — Kitchen Hygiene Monitor
**Before the presentation (10 min early):** Runtime → Change runtime type → T4 GPU, then run **cell 1**.
**In front of sir:** run **cell 2** (about 30–60 s on GPU). Cell 3 and 4 are optional.

## 1. Setup (run before presenting)

In [ ]:
!pip install -q ultralytics lap
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, json, base64, subprocess
BASE = '/content/drive/MyDrive/KitchenHygiene'
CODE, VIDEOS = f'{BASE}/code', f'{BASE}/videos'
OUT = '/content/demo_output'                       # demo writes here, so the main results stay untouched
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, CODE)
from kitchen_hygiene.config import Config
from kitchen_hygiene.pipeline import process_video
from kitchen_hygiene.perception import Perceiver
import pandas as pd
from IPython.display import display, HTML, Image

ZONES = {
    "sink":       [[0.05,0.38],[0.33,0.38],[0.33,0.90],[0.05,0.90]],
    "soap":       [[0.15,0.34],[0.24,0.34],[0.24,0.55],[0.15,0.55]],
    "prep":       [[0.36,0.52],[0.55,0.52],[0.55,0.75],[0.36,0.75]],
    "storage":    [[0.45,0.42],[0.56,0.42],[0.56,0.51],[0.45,0.51]],
    "used_oil":   [[0.56,0.46],[0.62,0.46],[0.62,0.64],[0.56,0.64]],
    "cooking":    [[0.61,0.54],[0.87,0.54],[0.87,0.93],[0.61,0.93]],
    "clean_rack": [[0.67,0.15],[0.96,0.15],[0.96,0.35],[0.67,0.35]],
    "used_rack":  [[0.66,0.36],[0.95,0.36],[0.95,0.58],[0.66,0.58]],
}
cfg = Config(); cfg.zones = ZONES; cfg.sample_fps = 5; cfg.wash_min_s = 6
custom = f'{BASE}/runs/kitchen416/weights/best.pt'
cfg.custom_models = [custom] if os.path.exists(custom) else []
perceiver = Perceiver(cfg)                         # loads the models once, so the live run is fast
print('Ready. Models loaded:', 'YOLOv8-pose + YOLOv8 COCO' + (' + our cockroach model' if cfg.custom_models else ''))
print('Clips:', sorted(os.path.basename(p) for p in glob.glob(f'{VIDEOS}/*.mp4')))

## 2. LIVE: run the system on one clip
Change `CLIP` to any clip name, e.g. `S05`, `S01`, `S12`, `H1`.

In [ ]:
CLIP = 'S05'
r = process_video(f'{VIDEOS}/{CLIP}.mp4', cfg, OUT, perceiver=perceiver)

rows = [dict(rule=f['vtype'], status=f['status'], confidence=round(f['confidence'], 2),
             time=f"{f['t_start']:.1f}–{f['t_end']:.1f} s", severity=f.get('severity', ''), reason=f['reason'])
        for f in r['findings']]
pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(rows) if rows else 'No findings')

for f in r['findings']:
    if f['status'] == 'violation' and f.get('keyframe'):
        print(f"Evidence frame for {f['vtype']} (confidence {f['confidence']:.2f}):")
        display(Image(f['keyframe'], width=640))
        break

## 3. Play the annotated video (zones, person ID, wrists, alerts)

In [ ]:
src = r['annotated']; dst = f'{OUT}/{CLIP}_play.mp4'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', dst])
b64 = base64.b64encode(open(dst, 'rb').read()).decode()
display(HTML(f'<video width="720" controls autoplay><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 4. Why did it decide that? (the soap Bayesian network for this clip)

In [ ]:
for f in r['findings']:
    if f['vtype'] == 'HANDWASH_VIOLATION':
        print(f"{f['status'].upper()} · {f['reason']}")
        print(json.dumps(f.get('evidence', {}), indent=2, default=str)[:1500])

## 5. The manager's dashboard
Everything the system found in this session (run cell 2 on a few clips first to fill it up).

In [ ]:
from kitchen_hygiene.dashboard import build_dashboard
if 'all_findings' not in globals(): all_findings = {}
all_findings[CLIP] = r['findings']
dash = build_dashboard([f for fs in all_findings.values() for f in fs], f'{OUT}/dashboard.html',
                       title='Kitchen Hygiene Monitor', period='Demo session', rel_to=OUT)
display(HTML(open(f'{OUT}/dashboard.html').read()))